# 1. Set up a local database

**Start here.** This notebook gets a local biodata registry running on your machine and seeds it with example records, so the other notebooks have something to talk to. The follow-on notebooks all assume the steps below have been done at least once.

Once the stack is up and populated, move on to:

- [`2_Create_and_Update_Metadata.ipynb`](./2_Create_and_Update_Metadata.ipynb) — creating, updating, and deleting metadata records via the Python client.
- [`Integrated_Document_Analysis.ipynb`](./Integrated_Document_Analysis.ipynb) — exploratory analysis against the joined document store (currently on the `docs/integrated-document-analysis` branch; will join this set once merged).

## Running the services with Docker

From the `biodata-registry-sandbox` folder:

```
docker compose up --build -d
```

Containers spin up in the background; it may take a minute. Check status with `docker ps`. Expected services:

```
CONTAINER ID   IMAGE                             NAMES
...            biodata-registry-sandbox-lambda   lambda
...            quay.io/debezium/connect:3.1      biodata-registry-sandbox-connect-1
...            quay.io/debezium/kafka:3.1        biodata-registry-sandbox-kafka-1
...            biodata-registry-sandbox-api      registry_api
...            mongo:latest                      mongodb
...            quay.io/debezium/zookeeper:3.1    biodata-registry-sandbox-zookeeper-1
...            debezium/postgres:15              biodata-registry-sandbox-postgres-1
```

The `lambda` service is the last to come up; once it's running, the stack is ready.

## Install the client

There is an auto-generated Python client used to interact with the API server. From the `client/` folder, in your Python environment:

```
pip install -e .
```

## Populate the databases

The cell below populates Postgres (and, via the lambda, MongoDB) with ~400 records pulled from a DocDB sample.
This takes a minute or two; a progress bar will appear once the records start loading.

In [2]:
%run populate_database.py

Populating registry: 100%|██████████| 400/400 [00:25<00:00, 15.61record/s]


## Smoke test

A quick check that the API is reachable and that the populate step worked. If the call below returns records, you're ready to move on to the other notebooks.

In [3]:
from biodata_registry_api_client import Configuration, ApiClient, CoreApi

core_api = CoreApi(ApiClient(Configuration(host="http://localhost:5000")))
subjects = core_api.get_subjects(limit=5, offset=0)
print(f"API is up. {len(subjects)} subjects fetched (showing first):")
print(subjects[0])

API is up. 5 subjects fetched (showing first):
id=1 name='710240' data={'notes': " (SubjectUpgraderV1V2): Source not specified, defaulting to 'Other'.", 'subject_id': '710240', 'describedBy': 'https://raw.githubusercontent.com/AllenNeuralDynamics/aind-data-schema/main/src/aind_data_schema/core/subject.py', 'object_type': 'Subject', 'schema_version': '2.2.1', 'subject_details': {'sex': 'Male', 'rrid': None, 'source': {'name': 'Other', 'registry': None, 'abbreviation': None, 'registry_identifier': None}, 'strain': {'name': 'C57BL/6J', 'species': 'Mus musculus', 'registry': 'Mouse Genome Informatics (MGI)', 'registry_identifier': 'MGI:3028467'}, 'alleles': [], 'housing': None, 'species': {'name': 'Mus musculus', 'registry': 'National Center for Biotechnology Information (NCBI)', 'common_name': 'House mouse', 'registry_identifier': 'NCBI:txid10090'}, 'genotype': 'wt/wt ', 'object_type': 'Mouse subject', 'restrictions': None, 'breeding_info': {'maternal_id': 'None', 'object_type': 'Breeding